# Fine-tune a Small LLM (Qwen2.5-0.5B) for Named Entity Recognition

we treat NER as an **instruction-following / text generation** task and fine-tune a small causal LLM (`Qwen/Qwen2.5-0.5B-Instruct`) using **LoRA (PEFT)**.

**Input:** a sentence  
**Output:** JSON list of `{text, type}` entities

Dataset: `ner_dataset_50k.csv` (columns: `Input`, `Output`)

## 1. Install dependencies (run once)

In [1]:
%pip install -q transformers datasets peft accelerate bitsandbytes trl pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 39.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## 2. Load & inspect the dataset

In [2]:
import pandas as pd, json, ast, random

df = pd.read_csv(r'/kaggle/input/datasets/mirajverisk/ner-daata/ner_dataset_50k.csv')
print('Shape:', df.shape)
df.head(3)

Shape: (50000, 2)


,Input,Output
0,Taylor Swift from Tesla will speak at Grammy A...,"[{""text"": ""Taylor Swift"", ""type"": ""PERSON""}, {..."
1,Vladimir Putin visited Mount Everest on late 2...,"[{""text"": ""Vladimir Putin"", ""type"": ""PERSON""},..."
2,Vladimir Putin earned 500 AUD by selling Midjo...,"[{""text"": ""Vladimir Putin"", ""type"": ""PERSON""},..."


## 3. Build instruction-style prompts (Qwen chat template)

We use a small subset for speed. Bump `SAMPLE_SIZE` for better quality.

In [3]:
SAMPLE_SIZE = 5000  # increase toward 50000 for full training
df_small = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)

from sklearn.model_selection import train_test_split
train_df, eval_df = train_test_split(df_small, test_size=0.2, random_state=42)
print(len(train_df), len(eval_df))

4000 1000


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM_PROMPT = (
    'You are an NER model. Extract named entities from the sentence and '
    'return ONLY a JSON list of objects with keys "text" and "type". '
    'Allowed types: PERSON, ORGANIZATION, LOCATION, DATE, EVENT, PRODUCT, '
    'MONEY, TIME, WORK_OF_ART, LANGUAGE, NORP, FAC, GPE.'
)

def build_messages(sentence, answer=None):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': sentence},
    ]
    if answer is not None:
        msgs.append({'role': 'assistant', 'content': answer})
    return msgs

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
messages =  build_messages(train_df.iloc[0]['Input'], train_df.iloc[0]['Output'])
messages 

[{'role': 'system',
  'content': 'You are an NER model. Extract named entities from the sentence and return ONLY a JSON list of objects with keys "text" and "type". Allowed types: PERSON, ORGANIZATION, LOCATION, DATE, EVENT, PRODUCT, MONEY, TIME, WORK_OF_ART, LANGUAGE, NORP, FAC, GPE.'},
 {'role': 'user',
  'content': 'Angela Merkel moved to Taj Mahal in September 11 and started working at Cambridge University.'},
 {'role': 'assistant',
  'content': '[{"text": "Angela Merkel", "type": "PERSON"}, {"text": "Taj Mahal", "type": "LOCATION"}, {"text": "September 11", "type": "DATE"}, {"text": "Cambridge University", "type": "ORGANIZATION"}]'}]

In [6]:
# Preview one
sample_txt = tokenizer.apply_chat_template(
    messages,
    tokenize=False
)
sample_txt

'<|im_start|>system\nYou are an NER model. Extract named entities from the sentence and return ONLY a JSON list of objects with keys "text" and "type". Allowed types: PERSON, ORGANIZATION, LOCATION, DATE, EVENT, PRODUCT, MONEY, TIME, WORK_OF_ART, LANGUAGE, NORP, FAC, GPE.<|im_end|>\n<|im_start|>user\nAngela Merkel moved to Taj Mahal in September 11 and started working at Cambridge University.<|im_end|>\n<|im_start|>assistant\n[{"text": "Angela Merkel", "type": "PERSON"}, {"text": "Taj Mahal", "type": "LOCATION"}, {"text": "September 11", "type": "DATE"}, {"text": "Cambridge University", "type": "ORGANIZATION"}]<|im_end|>\n'

## 4. Tokenize with prompt masking
We mask the prompt tokens so the loss is computed only on the assistant's answer.

In [7]:
print(tokenizer.model_max_length)

131072


In [8]:
import numpy as np

# Sample 5,000 examples (change if you want)
sample_df = df.sample(n=min(5000, len(df)), random_state=42)

texts = [
    tokenizer.apply_chat_template(
        build_messages(inp, out),
        tokenize=False,
        add_generation_prompt=False,
    )
    for inp, out in zip(sample_df["Input"], sample_df["Output"])
]

encodings = tokenizer(texts, add_special_tokens=False)

lengths = [len(ids) for ids in encodings["input_ids"]]

print(f"Min: {np.min(lengths)}")
print(f"Mean: {np.mean(lengths):.2f}")
print(f"Median: {np.median(lengths)}")
print(f"90th percentile: {np.percentile(lengths, 90)}")
print(f"95th percentile: {np.percentile(lengths, 95)}")
print(f"99th percentile: {np.percentile(lengths, 99)}")
print(f"Max: {np.max(lengths)}")

for max_len in [256, 512, 1024, 2048]:
    truncated = sum(l > max_len for l in lengths)
    print(
        f"MAX_LEN={max_len}: "
        f"{truncated}/{len(lengths)} "
        f"({truncated / len(lengths) * 100:.2f}%) examples would be truncated"
    )

Min: 118
Mean: 146.58
Median: 144.0
90th percentile: 165.0
95th percentile: 170.0
99th percentile: 177.0
Max: 187
MAX_LEN=256: 0/5000 (0.00%) examples would be truncated
MAX_LEN=512: 0/5000 (0.00%) examples would be truncated
MAX_LEN=1024: 0/5000 (0.00%) examples would be truncated
MAX_LEN=2048: 0/5000 (0.00%) examples would be truncated


In [9]:
from datasets import Dataset

MAX_LEN = 256

def encode(example):

    prompt_ids = tokenizer.apply_chat_template(
        build_messages(example['Input']),
        tokenize=True,
        add_generation_prompt=True
    )

    prompt_ids = prompt_ids["input_ids"]


    full_ids = tokenizer.apply_chat_template(
        build_messages(example['Input'], example['Output']),
        tokenize=True
    )

    full_ids = full_ids["input_ids"]


    full_ids = full_ids[:MAX_LEN]


    prompt_len = min(len(prompt_ids), len(full_ids))

    labels = [-100] * prompt_len
    labels = labels + full_ids[prompt_len:]

    return {
        "input_ids": full_ids,
        "labels": labels,
        "attention_mask": [1] * len(full_ids) #since we dont have padding yet
    }


In [10]:
train_ds = Dataset.from_pandas(train_df[['Input','Output']]).map(encode)

eval_ds = Dataset.from_pandas(eval_df[['Input','Output']]).map(encode)

print(train_ds)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['Input', 'Output', '__index_level_0__', 'input_ids', 'labels', 'attention_mask'],
    num_rows: 4000
})


In [11]:
# def collate(batch):
#     maxlen = max(len(b['input_ids']) for b in batch)
#     def pad(seq, val):
#         return seq + [val] * (maxlen - len(seq))
#     input_ids = torch.tensor([pad(b['input_ids'], tokenizer.pad_token_id) for b in batch])
#     labels    = torch.tensor([pad(b['labels'], -100) for b in batch])
#     attn      = torch.tensor([pad(b['attention_mask'], 0) for b in batch])
#     return {'input_ids': input_ids, 'labels': labels, 'attention_mask': attn}

# Understanding the Data Collator

### Before Collator

```python
[
    {
        "input_ids": [1, 2, 3, 4],
        "labels": [-100, -100, 3, 4],
        "attention_mask": [1, 1, 1, 1]
    },
    {
        "input_ids": [1, 2],
        "labels": [-100, 2],
        "attention_mask": [1, 1]
    }
]
```

### After Collator

```python
input_ids:
[
    [1, 2, 3, 4],
    [1, 2, pad, pad]
]

labels:
[
    [-100, -100, 3, 4],
    [-100, 2, -100, -100]
]

attention_mask:
[
    [1, 1, 1, 1],
    [1, 1, 0, 0]
]
```

In [12]:
from transformers import DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True
)

## 5. Load model + attach LoRA adapters

In [13]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.0 MB/s eta 0:00:00


In [14]:
from peft import LoraConfig, get_peft_model, TaskType

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if device == 'cuda' else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=dtype, trust_remote_code=True
).to(device)

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


## 6. Train

In [15]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir='./qwen_ner_lora',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    logging_steps=25,
    save_steps=25,
    eval_strategy='steps',
    eval_steps=25,
    save_total_limit=2,
    fp16=(device=='cuda'),
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=collator, processing_class=tokenizer,
)
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
25,0.066765,0.005753
50,0.003087,0.001207
75,0.000722,0.000705
100,0.000458,0.000546
125,0.000422,0.000531


TrainOutput(global_step=125, training_loss=0.014290573097765446, metrics={'train_runtime': 380.514, 'train_samples_per_second': 10.512, 'train_steps_per_second': 0.329, 'total_flos': 1437109230796800.0, 'train_loss': 0.014290573097765446, 'epoch': 1.0})

In [16]:
model.save_pretrained('./qwen_ner/final')
tokenizer.save_pretrained('./qwen_ner/final')
print('Saved LoRA adapters')

Saved LoRA adapters


## 7. Inference

In [17]:
def predict(sentence, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(
        build_messages(sentence), tokenize=False, add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.pad_token_id,
        )
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    try:
        return json.loads(gen)
    except Exception:
        return gen

tests = [
    'Elon Musk visited Berlin last Monday to open a new Tesla factory.',
    'The Champions League final will be held in Istanbul in June 2025.',
    'Apple announced the iPhone 17 at their September event in Cupertino.',
]

for t in tests:
    print(t)
    print(' ->', predict(t))
    print()

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Elon Musk visited Berlin last Monday to open a new Tesla factory.
 -> [{'text': 'Elon Musk', 'type': 'PERSON'}, {'text': 'Berlin', 'type': 'LOCATION'}, {'text': 'Monday', 'type': 'DATE'}, {'text': 'Tesla factory', 'type': 'PRODUCT'}]

The Champions League final will be held in Istanbul in June 2025.
 -> [{'text': 'Champions League', 'type': 'EVENT'}, {'text': 'Istanbul', 'type': 'LOCATION'}, {'text': 'June 2025', 'type': 'DATE'}]

Apple announced the iPhone 17 at their September event in Cupertino.
 -> [{'text': 'Apple', 'type': 'ORGANIZATION'}, {'text': 'iPhone 17', 'type': 'PRODUCT'}, {'text': 'September', 'type': 'DATE'}, {'text': 'Cupertino', 'type': 'LOCATION'}]



In [18]:
 predict("Elon Musk Works at Google He Lives in Kathmandu and Play Football")

[{'text': 'Elon Musk', 'type': 'PERSON'},
 {'text': 'Google', 'type': 'ORGANIZATION'},
 {'text': 'Kathmandu', 'type': 'LOCATION'},
 {'text': 'Play Football', 'type': 'EVENT'}]

## 8. Simple entity-level evaluation on the eval split

In [19]:
import json
import ast

def to_set(ents):
    # Parse if it's a string
    if isinstance(ents, str):
        try:
            ents = json.loads(ents)
        except Exception:
            try:
                ents = ast.literal_eval(ents)
            except Exception:
                return set()

    if not isinstance(ents, list):
        return set()

    return {
        (
            e.get("text", "").strip().lower(),
            e.get("type", "").strip().upper()
        )
        for e in ents
        if isinstance(e, dict)
    }

N = 100  # subset for speed
tp = fp = fn = 0

for _, row in eval_df.head(N).iterrows():
    pred = predict(row["Input"])

    P = to_set(pred)
    G = to_set(row["Output"])

    tp += len(P & G)
    fp += len(P - G)
    fn += len(G - P)

prec = tp / (tp + fp) if (tp + fp) else 0
rec  = tp / (tp + fn) if (tp + fn) else 0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0

print(f"Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}")

Precision=1.000  Recall=1.000  F1=1.000


# Upload to the Hugging Face

In [20]:
# !pip install huggingface_hub -q

In [21]:
# from huggingface_hub import login, HfApi

# login()

In [22]:
# # Upload the folder
# api = HfApi()

# api.upload_folder(
#     folder_path="./qwen_ner_lora/final",
#     repo_id="mirajbhandari/Entity_Extcation_Quen",
#     repo_type="model",
# )

In [23]:
# from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# from peft import PeftModel

# BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# ADAPTER = "mirajbhandari/Entity_Extcation_Quen"

# tokenizer = AutoTokenizer.from_pretrained(ADAPTER)

# base_model = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL,
#     device_map="auto",
#     torch_dtype="auto"
# )

# model = PeftModel.from_pretrained(
#     base_model,
#     ADAPTER
# )


In [24]:
# prompt = tokenizer.apply_chat_template(
#     build_messages("Miraj is good boy. he lives in kathmandu. he loves to play the guitar"),
#     tokenize=False,
#     add_generation_prompt=True
# )

# result = pipe(
#     prompt,
#     max_new_tokens=128,
#     do_sample=False,
#     return_full_text=False,  # only return the generated response
# )

# print(result[0]["generated_text"])